# **5일차 과제 — 진료 데이터 기반 QA 챗봇 구현하기1 (텍스트 기반)**

* 4일차에서 구축한 만성질환 벡터DB(`./chronic_db`)를 활용해서, **RAG를 도구로 사용하는 방식**과 **RAG를 노드로 사용하는 방식**의 QA 챗봇을 각각 구현하고, 동일한 질문에 대한 두 챗봇의 답변을 비교합니다.
* ⚠️ 4일차에서 생성된 `chronic_db` 폴더를 이번 노트북과 같은 위치(구글 드라이브 `ai_agent` 폴더)에 그대로 두고 실행하세요.

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 5일차 과제 (RAG 기초2 · Agentic RAG)

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키·벡터DB 없이도 돌아가는 셀(검색 도구 정의, State·노드 함수 정의, 두 챗봇의 그래프 조립·구조 확인)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 OpenAI API 키와 4일차 벡터DB가 필요해서 아직 실행하지 않았습니다.
- Colab 준비물: 구글 드라이브 `MyDrive/ai_agent/` 폴더에 **`api_key.txt`**와 4일차에서 만든 **`chronic_db` 폴더**를 넣어 두고 **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요. (키 파일 대신 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY`를 등록해도 됩니다.)
- GPT 답변은 실행할 때마다 조금씩 달라질 수 있습니다.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시)
!pip install langchain-openai langgraph langchain-chroma -q

In [1]:
import os   # 환경변수(API 키), 파일 경로 관리

from langchain_openai import ChatOpenAI, OpenAIEmbeddings               # GPT 모델 / 임베딩 모델
from langchain_core.messages import HumanMessage, SystemMessage         # 메시지 종류
from langchain_core.tools import tool                                   # 함수를 LLM용 '도구'로 만드는 데코레이터
from langchain_chroma import Chroma                                     # 벡터DB(Chroma)
from langgraph.graph import StateGraph, START, END, MessagesState       # 그래프 도구 + messages 칸이 있는 기본 State
from langgraph.prebuilt import ToolNode, tools_condition                # 도구 실행 노드, 도구 호출 여부로 분기하는 함수

In [ ]:
# 🔑 [Colab에서 실행] 구글드라이브 연결 (api_key.txt와 chronic_db가 드라이브에 있음)
from google.colab import drive   # 구글 드라이브 연결 도구
drive.mount('/content/drive')    # 드라이브를 /content/drive 경로에 연결 (처음엔 권한 허용 창이 뜸)

## **(2) API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] 드라이브의 api_key.txt에서 API 키를 읽어 환경변수로 등록합니다
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록

path = '/content/drive/MyDrive/ai_agent/'               # api_key.txt와 chronic_db를 넣어 둔 드라이브 폴더

# API 키 로드 및 환경변수 설정
if os.path.exists(path + 'api_key.txt'):                # 드라이브에 키 파일이 있으면
    load_api_keys(path + 'api_key.txt')                 # 파일에서 읽어서 등록
ensure_api_key('OPENAI_API_KEY')                        # 그래도 없으면 보안 비밀/직접 입력으로 채움

* 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")

## **(3) 벡터DB 연결**
* 4일차에서 구축한 `chronic_db`를 그대로 불러옵니다.

In [ ]:
# 🔑 [Colab에서 실행] 4일차 벡터DB 불러오기 (임베딩 모델 생성에 API 키 필요)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")   # ⚠️ 4일차에서 DB를 만들 때와 '같은' 임베딩 모델이어야 함

vectorstore = Chroma(embedding_function=embedding,
                      persist_directory= path + "./chronic_db")   # 드라이브의 ai_agent/chronic_db 폴더를 연결

print("불러온 청크 수:", vectorstore._collection.count())   # 0이면 경로가 틀렸거나 4일차 DB가 없는 것

# **2. QA 챗봇① — RAG를 도구로 사용**

**💡 풀이 포인트 (RAG를 도구로)** — 벡터DB 검색을 `@tool` 함수로 감싸서 LLM에게 "필요하면 쓰라"고 쥐여 주는 방식입니다. LLM이 질문을 보고 **검색할지 말지, 어떤 검색어로 찾을지**를 스스로 정하고(`tool_calls`), 3일차와 같은 `ToolNode` + `tools_condition` 구조로 실행됩니다. 도구 docstring에 "DB에 어떤 정보가 들어 있는지"를 구체적으로 적어야 LLM이 알맞은 때에 검색을 요청합니다.

## **(1) 검색 도구 정의**

In [2]:
# [챗봇①] 벡터DB 검색을 LLM이 쓸 수 있는 '도구'로 만듭니다 (docstring = LLM이 읽는 도구 설명서)
@tool
def search_chronic_db(query: str) -> str:
    """만성질환 정보 벡터DB에서 질문과 관련된 문서를 검색합니다.
    고혈압, 제2형 당뇨병, 고지혈증, 골다공증, 천식, 만성신장질환의
    증상과 진단 기준, 생활습관 관리 방법, 복약 및 주의사항에 대한 질문에 사용하세요.

    Args:
        query: 검색할 질문이나 키워드 (예: "고혈압 식단 주의사항")
    """
    docs = vectorstore.similarity_search(query, k=3)              # 의미가 가장 가까운 청크 3개 검색
    return "\n\n---\n\n".join(doc.page_content for doc in docs)   # 청크 내용을 구분선으로 이어 하나의 문자열로 반환


# 도구가 LLM에게 어떻게 보이는지 확인 (DB 연결 없이도 확인 가능)
print("도구 이름:", search_chronic_db.name)
print("인자 정보:", search_chronic_db.args)

도구 이름: search_chronic_db
인자 정보: {'query': {'title': 'Query', 'type': 'string'}}


## **(2) Tool Node, State, LLM 준비**

In [3]:
tools = [search_chronic_db]   # LLM에게 줄 도구 목록
tool_node = ToolNode(tools)   # LLM이 도구 호출을 요청하면 실제로 실행해 주는 노드

# State: MessagesState를 그대로 사용합니다 (messages 칸 하나에 질문·도구 호출·검색 결과·답변이 모두 쌓임)

In [ ]:
# 🔑 [Colab에서 실행] LLM 준비 + 도구 연결 (API 키 필요)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # temperature=0 → 답이 흔들리지 않게
llm_with_tools = llm.bind_tools(tools)                  # LLM에게 검색 도구 설명서를 쥐여 줌

## **(3) 노드 준비**

In [4]:
def chatbot_tool(state: MessagesState):
    # 검색 여부는 LLM이 판단: "필요하면" 도구를 쓰라고만 안내합니다
    s_msg = SystemMessage(content="당신은 만성질환 환자를 돕는 병원 QA 챗봇입니다. "
                                  "만성질환 관련 정보가 필요하면 search_chronic_db 도구로 DB를 검색하고, "
                                  "검색했다면 그 결과에 근거해 답하세요. 환자가 이해하기 쉬운 한국어로 답하세요.")
    response = llm_with_tools.invoke([s_msg] + state["messages"])   # 도구 호출 요청 또는 최종 답변
    return {"messages": [response]}                                  # 응답을 messages에 쌓기

## **(4) 그래프 구성**

In [5]:
builder = StateGraph(MessagesState)                       # messages 기본 State
builder.add_node("chatbot", chatbot_tool)                 # LLM 판단 노드
builder.add_node("tools", tool_node)                      # 검색 도구 실행 노드 (이름 "tools" → tools_condition과 맞춤)
builder.add_edge(START, "chatbot")                        # 시작 → chatbot
builder.add_conditional_edges("chatbot", tools_condition) # 도구 호출 요청이 있으면 tools, 없으면 END
builder.add_edge("tools", "chatbot")                      # 검색 결과를 들고 다시 chatbot으로 → 최종 답변
tool_graph = builder.compile()                            # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = tool_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → chatbot 
chatbot → __end__    (조건부)
chatbot → tools    (조건부)
tools → chatbot 


## **(5) 실행**

In [ ]:
# 🔑 [Colab에서 실행] 챗봇① 실행 (API 키 + 벡터DB 필요)
result_tool = tool_graph.invoke({"messages": [HumanMessage(content="고혈압 환자가 주의해야 할 음식은?")]})   # 질문을 넣어 실행

for m in result_tool["messages"]:   # Human → AI(검색 요청) → Tool(검색 결과) → AI(최종 답변) 흐름 확인
    m.pretty_print()                # 메시지 종류별로 보기 좋게 출력

# **3. QA 챗봇② — RAG를 노드로 사용**

**💡 풀이 포인트 (RAG를 노드로)** — 이번에는 검색을 그래프의 고정된 단계(`retrieve` 노드)로 넣어서, **어떤 질문이든 항상 먼저 검색**한 뒤 `generate` 노드가 검색 결과(context)를 근거로 답합니다. LLM이 검색 여부를 고르지 않으니 흐름이 단순하고 예측 가능하지만, 사용자 질문 문장을 그대로 검색어로 쓰고 검색이 필요 없는 질문에도 검색한다는 차이가 있습니다.

## **(1) DB 검색 함수 준비**

In [6]:
# [챗봇②] 도구가 아닌 '일반 함수'로 검색 기능을 준비합니다 (노드 안에서 직접 호출)
def search_db(query: str, k: int = 3) -> str:
    docs = vectorstore.similarity_search(query, k=k)              # 질문과 가장 가까운 청크 k개 검색
    return "\n\n---\n\n".join(doc.page_content for doc in docs)   # 하나의 문자열(context)로 합치기

## **(2) State, LLM 준비**

In [7]:
class RAGState(MessagesState):   # MessagesState(messages 칸)를 물려받고 칸을 더 추가
    question: str                # 사용자 질문 (= 검색어)
    context: str                 # 검색된 문서 내용
    answer: str                  # 최종 답변

In [ ]:
# 🔑 [Colab에서 실행] LLM 준비 (API 키 필요) — 챗봇①의 llm을 써도 되지만, 이 절만 따로 실행해도 되도록 다시 준비
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # 답변 생성용 GPT

## **(3) 노드 준비**

In [8]:
def retrieve(state: RAGState):
    question = state["messages"][-1].content   # 가장 최근 사용자 메시지 = 질문
    context = search_db(question)              # ⭐ 질문이 오면 '항상' 검색 (LLM이 고르지 않음)
    return {"question": question, "context": context}


def generate(state: RAGState):
    s_msg = SystemMessage(content="당신은 만성질환 환자를 돕는 병원 QA 챗봇입니다. "
                                  "아래 [참고 문서]에 근거해서만 답하고, 문서에 없는 내용은 추측하지 말고 모른다고 답하세요. "
                                  "환자가 이해하기 쉬운 한국어로 답하세요.")
    h_msg = HumanMessage(content=f"[참고 문서]\n{state['context']}\n\n[질문]\n{state['question']}")   # 검색 결과 + 질문
    response = llm.invoke([s_msg, h_msg])                    # 검색 결과를 근거로 답변 생성
    return {"messages": [response], "answer": response.content}

## **(4) 그래프 구성**

In [9]:
builder = StateGraph(RAGState)              # 질문·문서·답변 칸이 있는 State
builder.add_node("retrieve", retrieve)      # 검색 노드 (항상 실행)
builder.add_node("generate", generate)      # 답변 생성 노드
builder.add_edge(START, "retrieve")         # 시작 → 검색
builder.add_edge("retrieve", "generate")    # 검색 → 답변 생성 (분기 없이 일직선)
builder.add_edge("generate", END)           # 답변 생성 → 끝
node_graph = builder.compile()              # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = node_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → retrieve 
retrieve → generate 
generate → __end__ 


## **(5) 실행**

In [ ]:
# 🔑 [Colab에서 실행] 챗봇② 실행 (API 키 + 벡터DB 필요)
result_node = node_graph.invoke({"messages": [HumanMessage(content="고혈압 환자가 주의해야 할 음식은?")]})   # 질문을 넣어 실행

print("[검색된 문서 일부]\n", result_node["context"][:300], "...\n")   # 항상 검색된 context 확인
print("[답변]\n", result_node["answer"])

# **4. 두 방식 비교**

* 동일한 질문 3개를 두 챗봇에 각각 실행해서, 답변 내용과 방식(tool 호출 여부 등)을 비교합니다.

**💡 풀이 포인트** — 같은 질문 3개를 두 챗봇에 똑같이 넣고, 답변과 함께 **도구 호출 여부와 실제로 쓴 검색어**를 기록해서 비교합니다. 마지막에 표(pandas DataFrame)로 모아 보면 차이가 한눈에 보입니다.

In [10]:
test_questions = [
    "고혈압 환자가 주의해야 할 음식은?",
    "당뇨병 진단 기준이 어떻게 되나요?",
    "골다공증 환자는 어떤 운동을 해야 하나요?",
]

In [11]:
def ask_tool_bot(question):
    """챗봇①(RAG-as-tool) 실행 → 답변, 도구 호출 여부, LLM이 만든 검색어를 돌려줌"""
    result = tool_graph.invoke({"messages": [HumanMessage(content=question)]})   # 챗봇① 실행
    queries = [call["args"].get("query")                         # LLM이 도구에 넘긴 검색어
               for m in result["messages"]                       # 모든 메시지 중에서
               for call in (getattr(m, "tool_calls", None) or [])]   # 도구 호출 요청이 있는 것만
    return {"answer": result["messages"][-1].content,            # 마지막 메시지 = 최종 답변
            "used_tool": len(queries) > 0,                       # 검색어가 하나라도 있으면 도구를 쓴 것
            "queries": queries}


def ask_node_bot(question):
    """챗봇②(RAG-as-node) 실행 → 답변과 검색어(= 질문 그대로)를 돌려줌"""
    result = node_graph.invoke({"messages": [HumanMessage(content=question)]})   # 챗봇② 실행
    return {"answer": result["answer"], "query": result["question"]}

In [ ]:
# 🔑 [Colab에서 실행] 질문 3개를 두 챗봇에 각각 실행해서 나란히 출력 (API 키 + 벡터DB 필요)
comparison = []   # 나중에 표로 정리할 결과 모음
for q in test_questions:
    tool_res = ask_tool_bot(q)   # ① RAG-as-tool
    node_res = ask_node_bot(q)   # ② RAG-as-node
    comparison.append({"질문": q, "tool": tool_res, "node": node_res})   # 표로 만들 결과 저장

    print("=" * 90)
    print("❓ 질문:", q)
    print(f"\n[① RAG-as-tool] 도구 호출: {'예' if tool_res['used_tool'] else '아니오'} | 검색어: {tool_res['queries']}")
    print(tool_res["answer"])
    print(f"\n[② RAG-as-node] 항상 검색 | 검색어: {node_res['query']}")
    print(node_res["answer"])

In [ ]:
# 🔑 [Colab에서 실행] 비교 결과를 표로 요약 (위 셀을 먼저 실행해야 함)
import pandas as pd   # 표(DataFrame)를 만드는 라이브러리

summary = pd.DataFrame([{   # 질문 하나당 한 줄씩 비교 결과를 담아 표 만들기
    "질문": c["질문"],
    "① 도구 호출": "예" if c["tool"]["used_tool"] else "아니오",
    "① 검색어(LLM이 만듦)": ", ".join(q for q in c["tool"]["queries"] if q),
    "② 검색어(질문 그대로)": c["node"]["query"],
    "① 답변 글자 수": len(c["tool"]["answer"]),
    "② 답변 글자 수": len(c["node"]["answer"]),
} for c in comparison])
summary   # 셀 마지막 줄에 변수만 쓰면 표가 보기 좋게 출력됨

**질문**: 두 방식의 답변 내용에 차이가 있나요? 있다면 왜 그런 차이가 발생했을지, RAG-as-tool과 RAG-as-node의 동작 방식 차이(항상 검색하는지 vs. 필요할 때만 검색하는지)와 연결지어 설명해보세요.

**답변 (조태희)**

> ✍️ 아래는 두 방식의 동작 원리에서 **예상되는 차이**를 정리한 것입니다. Colab에서 4절을 실행한 뒤 요약 표의 '도구 호출'·'검색어' 칸과 실제 답변을 보고, 예상과 같은 점·다른 점을 덧붙여 주세요.

두 챗봇의 답변은 큰 줄기는 비슷해도 세부 내용과 근거가 달라질 수 있습니다. 가장 큰 이유는 **검색을 언제, 어떤 검색어로 하느냐**입니다.

**RAG-as-node**는 질문이 들어오면 **항상** `retrieve` 노드에서 먼저 검색하고, 사용자 질문 문장을 그대로 검색어로 씁니다. 게다가 "참고 문서에 근거해서만 답하라"는 지침 때문에 답변이 DB 내용에 가깝게 묶입니다.

**RAG-as-tool**은 LLM이 **필요하다고 판단할 때만** 검색하고, 검색어도 LLM이 새로 만듭니다(질문을 짧은 키워드로 바꾸거나 여러 번 검색하기도 함). 그래서 같은 질문이라도 다른 청크가 검색될 수 있고, LLM이 이미 안다고 판단하면 검색 없이 자기 지식으로 답해서 DB에 없는 일반 상식이 섞이거나 반대로 DB의 구체적인 내용이 빠질 수 있습니다.

정리하면, 항상 검색하는 노드 방식은 **근거가 일정하고 예측 가능**한 대신 필요 없는 검색도 하고, 필요할 때만 검색하는 도구 방식은 **유연하고 효율적**인 대신 검색 여부와 검색어가 LLM 판단에 달려 있어 답변의 근거가 매번 달라질 수 있습니다. 병원 QA처럼 근거가 중요한 서비스에는 노드 방식이, 또는 도구 방식이라면 "의학 정보 질문은 반드시 검색하라"는 지침을 더한 형태가 더 안전하다고 생각합니다.